# Objective 3 — Step 2: Preprocessing Protocol

This notebook performs the following tasks before any model is trained:

1. Reload and clean the three credit-risk datasets.
2. Standardize the target as `1 = adverse/default/rejected`.
3. consolidate special category codes in the Taiwan dataset.
4. Verify repeated predictor profiles and target conflicts.
5. Create profile-group IDs to prevent identical profiles from crossing validation folds.
6. Create model-aware, leakage-free preprocessing pipelines.
7. Perform a one-fold preprocessing smoke test.
8. Save data dictionaries, feature mappings, split checks, and cleaned datasets.

**Do not train classification models in this notebook.**


In [1]:
%pip install pandas numpy scikit-learn openpyxl xlrd joblib


[notice] A new release of pip is available: 25.0.1 -> 26.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json
import platform
import re
import sys
import warnings

import joblib
import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE_DIR = Path(r"D:\PHD\Research Paper writing\3rd Obj. paper")

RESULTS_DIR = BASE_DIR / "results"
PREPROCESSING_DIR = RESULTS_DIR / "preprocessing_protocol"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "processed"
PIPELINE_DIR = BASE_DIR / "pipelines"

PREPROCESSING_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", BASE_DIR)
print("Preprocessing results:", PREPROCESSING_DIR)
print("Processed data:", PROCESSED_DATA_DIR)
print("Saved pipelines:", PIPELINE_DIR)


Project folder: D:\PHD\Research Paper writing\3rd Obj. paper
Preprocessing results: D:\PHD\Research Paper writing\3rd Obj. paper\results\preprocessing_protocol
Processed data: D:\PHD\Research Paper writing\3rd Obj. paper\data\processed
Saved pipelines: D:\PHD\Research Paper writing\3rd Obj. paper\pipelines


## 1. Locate and load the original files

In [3]:
def locate_file(patterns):
    for pattern in patterns:
        matches = sorted(BASE_DIR.glob(pattern))
        if matches:
            return matches[0]

    available = [path.name for path in BASE_DIR.iterdir()]
    raise FileNotFoundError(
        f"No matching file was found for {patterns}.\n"
        f"Available files: {available}"
    )


def clean_column_names(columns):
    cleaned = []
    for column in columns:
        column = str(column).strip()
        column = re.sub(r"[^A-Za-z0-9]+", "_", column)
        column = column.strip("_").lower()
        cleaned.append(column)
    return cleaned


AUSTRALIAN_FILE = locate_file([
    "*Australian_Credit_Approval*.csv",
    "*Australian*Credit*Approval*.csv",
])

GERMAN_FILE = locate_file([
    "*GermanCredit*.csv",
    "*German*Credit*.csv",
])

TAIWAN_FILE = locate_file([
    "*default*credit*card*clients*.xls",
    "*default*credit*card*clients*.xlsx",
])

print("Australian:", AUSTRALIAN_FILE.name)
print("German:", GERMAN_FILE.name)
print("Taiwan:", TAIWAN_FILE.name)


Australian: Australian_Credit_Approval.csv
German: GermanCredit.csv
Taiwan: default of credit card clients.xls


In [4]:
def read_taiwan_dataset(file_path):
    engine = "xlrd" if file_path.suffix.lower() == ".xls" else "openpyxl"

    for header_row in [1, 0]:
        dataframe = pd.read_excel(
            file_path,
            header=header_row,
            engine=engine,
        )
        dataframe.columns = clean_column_names(dataframe.columns)

        candidates = [
            column for column in dataframe.columns
            if "default" in column and "next" in column
        ]

        if candidates:
            return dataframe, candidates[0], header_row

    raise ValueError("Taiwan target column could not be identified.")


australian_raw = pd.read_csv(AUSTRALIAN_FILE)
australian_raw.columns = clean_column_names(australian_raw.columns)

german_raw = pd.read_csv(GERMAN_FILE)
german_raw.columns = clean_column_names(german_raw.columns)

taiwan_raw, TAIWAN_RAW_TARGET, taiwan_header_row = read_taiwan_dataset(
    TAIWAN_FILE
)

print("Australian raw shape:", australian_raw.shape)
print("German raw shape:", german_raw.shape)
print("Taiwan raw shape:", taiwan_raw.shape)
print("Taiwan original target:", TAIWAN_RAW_TARGET)
print("Taiwan header row used:", taiwan_header_row)


Australian raw shape: (690, 15)
German raw shape: (1000, 21)
Taiwan raw shape: (30000, 25)
Taiwan original target: default_payment_next_month
Taiwan header row used: 1


## 2. Define feature roles

In [5]:
FEATURE_DEFINITIONS = {
    "Australian Credit Approval": {
        "categorical": [
            "a1", "a4", "a5", "a6",
            "a8", "a9", "a11", "a12",
        ],
        "ordinal": [],
        "numerical": [
            "a2", "a3", "a7",
            "a10", "a13", "a14",
        ],
        "identifier": [],
        "raw_target": "class",
    },

    "German Credit": {
        "categorical": [
            "status",
            "credit_history",
            "purpose",
            "savings",
            "employment_duration",
            "personal_status_sex",
            "other_debtors",
            "property",
            "other_installment_plans",
            "housing",
            "job",
            "telephone",
            "foreign_worker",
        ],
        "ordinal": [],
        "numerical": [
            "duration",
            "amount",
            "installment_rate",
            "present_residence",
            "age",
            "number_credits",
            "people_liable",
        ],
        "identifier": [],
        "raw_target": "credit_risk",
    },

    "Taiwan Credit Card Default": {
        "categorical": [
            "sex",
            "education",
            "marriage",
        ],
        "ordinal": [
            "pay_0",
            "pay_2",
            "pay_3",
            "pay_4",
            "pay_5",
            "pay_6",
        ],
        "numerical": [
            "limit_bal",
            "age",
            "bill_amt1",
            "bill_amt2",
            "bill_amt3",
            "bill_amt4",
            "bill_amt5",
            "bill_amt6",
            "pay_amt1",
            "pay_amt2",
            "pay_amt3",
            "pay_amt4",
            "pay_amt5",
            "pay_amt6",
        ],
        "identifier": ["id"],
        "raw_target": TAIWAN_RAW_TARGET,
    },
}


## 3. Clean categories and create a unified adverse target

In [6]:
def map_with_unknown(series, mapping, prefix):
    mapped = series.map(mapping)

    unknown_mask = mapped.isna()
    if unknown_mask.any():
        mapped.loc[unknown_mask] = (
            prefix
            + "_unknown_code_"
            + series.loc[unknown_mask].astype(str)
        )

    return mapped.astype(str)


def prepare_australian(dataframe):
    df = dataframe.copy()

    df["class"] = pd.to_numeric(df["class"], errors="raise").astype(int)
    df["adverse_target"] = 1 - df["class"]

    categorical = FEATURE_DEFINITIONS[
        "Australian Credit Approval"
    ]["categorical"]

    for column in categorical:
        df[column] = "code_" + df[column].astype(str)

    return df


def prepare_german(dataframe):
    df = dataframe.copy()

    df["credit_risk"] = pd.to_numeric(
        df["credit_risk"],
        errors="raise",
    ).astype(int)

    df["adverse_target"] = 1 - df["credit_risk"]

    categorical = FEATURE_DEFINITIONS[
        "German Credit"
    ]["categorical"]

    for column in categorical:
        df[column] = df[column].astype(str).str.strip()

    return df


def prepare_taiwan(dataframe):
    df = dataframe.copy()

    df[TAIWAN_RAW_TARGET] = pd.to_numeric(
        df[TAIWAN_RAW_TARGET],
        errors="raise",
    ).astype(int)

    df["adverse_target"] = df[TAIWAN_RAW_TARGET]

    sex_mapping = {
        1: "male",
        2: "female",
    }

    education_mapping = {
        0: "other_or_unknown",
        1: "graduate_school",
        2: "university",
        3: "high_school",
        4: "other_or_unknown",
        5: "other_or_unknown",
        6: "other_or_unknown",
    }

    marriage_mapping = {
        0: "other_or_unknown",
        1: "married",
        2: "single",
        3: "other_or_unknown",
    }

    df["sex"] = map_with_unknown(
        df["sex"], sex_mapping, "sex"
    )
    df["education"] = map_with_unknown(
        df["education"], education_mapping, "education"
    )
    df["marriage"] = map_with_unknown(
        df["marriage"], marriage_mapping, "marriage"
    )

    return df


australian_df = prepare_australian(australian_raw)
german_df = prepare_german(german_raw)
taiwan_df = prepare_taiwan(taiwan_raw)

DATASETS = {
    "Australian Credit Approval": australian_df,
    "German Credit": german_df,
    "Taiwan Credit Card Default": taiwan_df,
}

for dataset_name, dataframe in DATASETS.items():
    values = set(dataframe["adverse_target"].unique())
    assert values.issubset({0, 1}), (
        f"{dataset_name} has invalid target values: {values}"
    )

    print("\n", dataset_name)
    print(dataframe["adverse_target"].value_counts().sort_index())



 Australian Credit Approval
adverse_target
0    307
1    383
Name: count, dtype: int64

 German Credit
adverse_target
0    700
1    300
Name: count, dtype: int64

 Taiwan Credit Card Default
adverse_target
0    23364
1     6636
Name: count, dtype: int64


## 4. Save the documented Taiwan category consolidation

In [7]:
category_mapping_table = pd.DataFrame([
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "sex",
        "original_codes": "1",
        "cleaned_category": "male",
        "reason": "Official category label",
    },
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "sex",
        "original_codes": "2",
        "cleaned_category": "female",
        "reason": "Official category label",
    },
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "education",
        "original_codes": "1",
        "cleaned_category": "graduate_school",
        "reason": "Official category label",
    },
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "education",
        "original_codes": "2",
        "cleaned_category": "university",
        "reason": "Official category label",
    },
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "education",
        "original_codes": "3",
        "cleaned_category": "high_school",
        "reason": "Official category label",
    },
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "education",
        "original_codes": "0, 4, 5, 6",
        "cleaned_category": "other_or_unknown",
        "reason": "Rare, other, or undocumented codes consolidated",
    },
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "marriage",
        "original_codes": "1",
        "cleaned_category": "married",
        "reason": "Official category label",
    },
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "marriage",
        "original_codes": "2",
        "cleaned_category": "single",
        "reason": "Official category label",
    },
    {
        "dataset": "Taiwan Credit Card Default",
        "variable": "marriage",
        "original_codes": "0, 3",
        "cleaned_category": "other_or_unknown",
        "reason": "Other or undocumented codes consolidated",
    },
])

category_mapping_table.to_csv(
    PREPROCESSING_DIR / "taiwan_category_consolidation.csv",
    index=False,
)

display(category_mapping_table)


,dataset,variable,original_codes,cleaned_category,reason
0,Taiwan Credit Card Default,sex,1,male,Official category label
1,Taiwan Credit Card Default,sex,2,female,Official category label
2,Taiwan Credit Card Default,education,1,graduate_school,Official category label
3,Taiwan Credit Card Default,education,2,university,Official category label
4,Taiwan Credit Card Default,education,3,high_school,Official category label
5,Taiwan Credit Card Default,education,"0, 4, 5, 6",other_or_unknown,"Rare, other, or undocumented codes consolidated"
6,Taiwan Credit Card Default,marriage,1,married,Official category label
7,Taiwan Credit Card Default,marriage,2,single,Official category label
8,Taiwan Credit Card Default,marriage,"0, 3",other_or_unknown,Other or undocumented codes consolidated


## 5. Verify repeated predictor profiles

Identical predictor profiles must not be split between training and test folds.  
A stable `profile_group_id` is therefore created from all modelling predictors.


In [8]:
def get_model_columns(dataset_name):
    definition = FEATURE_DEFINITIONS[dataset_name]

    return (
        definition["categorical"]
        + definition["ordinal"]
        + definition["numerical"]
    )


def add_profile_group_id(dataset_name, dataframe):
    model_columns = get_model_columns(dataset_name)

    group_hash = pd.util.hash_pandas_object(
        dataframe[model_columns],
        index=False,
    )

    result = dataframe.copy()
    result["profile_group_id"] = group_hash.astype("uint64").astype(str)

    return result


DATASETS = {
    dataset_name: add_profile_group_id(dataset_name, dataframe)
    for dataset_name, dataframe in DATASETS.items()
}

australian_df = DATASETS["Australian Credit Approval"]
german_df = DATASETS["German Credit"]
taiwan_df = DATASETS["Taiwan Credit Card Default"]


In [9]:
def duplicate_profile_analysis(dataset_name, dataframe):
    grouped = (
        dataframe
        .groupby("profile_group_id", dropna=False)
        .agg(
            group_size=("adverse_target", "size"),
            target_classes=("adverse_target", "nunique"),
            adverse_cases=("adverse_target", "sum"),
        )
        .reset_index()
    )

    grouped["favourable_cases"] = (
        grouped["group_size"] - grouped["adverse_cases"]
    )

    duplicate_groups = grouped[grouped["group_size"] > 1].copy()
    conflicting_groups = duplicate_groups[
        duplicate_groups["target_classes"] > 1
    ].copy()

    summary = {
        "dataset": dataset_name,
        "records": len(dataframe),
        "unique_predictor_profiles": grouped.shape[0],
        "duplicate_profile_groups": duplicate_groups.shape[0],
        "records_in_duplicate_groups": int(
            duplicate_groups["group_size"].sum()
        ),
        "conflicting_duplicate_groups": conflicting_groups.shape[0],
        "records_in_conflicting_groups": int(
            conflicting_groups["group_size"].sum()
        ),
        "decision": (
            "Retain all records; keep each predictor profile within one "
            "validation fold using profile_group_id."
        ),
    }

    slug = (
        dataset_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    duplicate_groups.to_csv(
        PREPROCESSING_DIR
        / f"{slug}_duplicate_profile_details.csv",
        index=False,
    )

    return summary


duplicate_summaries = [
    duplicate_profile_analysis(name, dataframe)
    for name, dataframe in DATASETS.items()
]

duplicate_profile_summary = pd.DataFrame(duplicate_summaries)

duplicate_profile_summary.to_csv(
    PREPROCESSING_DIR / "duplicate_profile_summary.csv",
    index=False,
)

display(duplicate_profile_summary)


,dataset,records,unique_predictor_profiles,duplicate_profile_groups,records_in_duplicate_groups,conflicting_duplicate_groups,records_in_conflicting_groups,decision
0,Australian Credit Approval,690,690,0,0,0,0,Retain all records; keep each predictor profil...
1,German Credit,1000,1000,0,0,0,0,Retain all records; keep each predictor profil...
2,Taiwan Credit Card Default,30000,29944,52,108,21,46,Retain all records; keep each predictor profil...


## 6. Build the final preprocessing data dictionary

In [10]:
def create_data_dictionary():
    rows = []

    for dataset_name, definition in FEATURE_DEFINITIONS.items():
        for role in [
            "categorical",
            "ordinal",
            "numerical",
            "identifier",
        ]:
            for variable in definition[role]:
                if role == "categorical":
                    linear_rule = (
                        "Most-frequent imputation, then one-hot encoding"
                    )
                    tree_rule = (
                        "Most-frequent imputation, then one-hot encoding"
                    )
                elif role == "ordinal":
                    linear_rule = (
                        "Most-frequent imputation, then standardization"
                    )
                    tree_rule = (
                        "Most-frequent imputation, retain ordinal values"
                    )
                elif role == "numerical":
                    linear_rule = (
                        "Median imputation, then standardization"
                    )
                    tree_rule = (
                        "Median imputation, retain original scale"
                    )
                else:
                    linear_rule = "Excluded from modelling"
                    tree_rule = "Excluded from modelling"

                note = ""

                if (
                    dataset_name == "Taiwan Credit Card Default"
                    and variable in {"sex", "age"}
                ):
                    note = "Retained for later subgroup fairness audit"

                if (
                    dataset_name == "Taiwan Credit Card Default"
                    and variable.startswith("pay_")
                ):
                    note = (
                        "Treated as ordered repayment-status information"
                    )

                rows.append({
                    "dataset": dataset_name,
                    "variable": variable,
                    "role": role,
                    "included_in_model": role != "identifier",
                    "linear_svm_knn_rule": linear_rule,
                    "tree_ensemble_rule": tree_rule,
                    "research_note": note,
                })

        rows.append({
            "dataset": dataset_name,
            "variable": definition["raw_target"],
            "role": "original_target",
            "included_in_model": False,
            "linear_svm_knn_rule": "Excluded from predictors",
            "tree_ensemble_rule": "Excluded from predictors",
            "research_note": "Retained only for traceability",
        })

        rows.append({
            "dataset": dataset_name,
            "variable": "adverse_target",
            "role": "unified_target",
            "included_in_model": False,
            "linear_svm_knn_rule": "Outcome: 1 = adverse/default",
            "tree_ensemble_rule": "Outcome: 1 = adverse/default",
            "research_note": "Common positive-class definition",
        })

        rows.append({
            "dataset": dataset_name,
            "variable": "profile_group_id",
            "role": "validation_group",
            "included_in_model": False,
            "linear_svm_knn_rule": "Excluded from predictors",
            "tree_ensemble_rule": "Excluded from predictors",
            "research_note": (
                "Keeps identical predictor profiles within one fold"
            ),
        })

    return pd.DataFrame(rows)


data_dictionary = create_data_dictionary()

data_dictionary.to_csv(
    PREPROCESSING_DIR / "preprocessing_data_dictionary.csv",
    index=False,
)

display(data_dictionary.head(20))


,dataset,variable,role,included_in_model,linear_svm_knn_rule,tree_ensemble_rule,research_note
0,Australian Credit Approval,a1,categorical,True,"Most-frequent imputation, then one-hot encoding","Most-frequent imputation, then one-hot encoding",
1,Australian Credit Approval,a4,categorical,True,"Most-frequent imputation, then one-hot encoding","Most-frequent imputation, then one-hot encoding",
2,Australian Credit Approval,a5,categorical,True,"Most-frequent imputation, then one-hot encoding","Most-frequent imputation, then one-hot encoding",
3,Australian Credit Approval,a6,categorical,True,"Most-frequent imputation, then one-hot encoding","Most-frequent imputation, then one-hot encoding",
4,Australian Credit Approval,a8,categorical,True,"Most-frequent imputation, then one-hot encoding","Most-frequent imputation, then one-hot encoding",
5,Australian Credit Approval,a9,categorical,True,"Most-frequent imputation, then one-hot encoding","Most-frequent imputation, then one-hot encoding",
6,Australian Credit Approval,a11,categorical,True,"Most-frequent imputation, then one-hot encoding","Most-frequent imputation, then one-hot encoding",
7,Australian Credit Approval,a12,categorical,True,"Most-frequent imputation, then one-hot encoding","Most-frequent imputation, then one-hot encoding",
8,Australian Credit Approval,a2,numerical,True,"Median imputation, then standardization","Median imputation, retain original scale",
9,Australian Credit Approval,a3,numerical,True,"Median imputation, then standardization","Median imputation, retain original scale",


## 7. Create model-aware preprocessing pipelines

In [11]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32,
        )


def build_preprocessor(dataset_name, mode):
    definition = FEATURE_DEFINITIONS[dataset_name]

    categorical = definition["categorical"]
    ordinal = definition["ordinal"]
    numerical = definition["numerical"]

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "onehot",
            make_one_hot_encoder(),
        ),
    ])

    if mode == "linear":
        numerical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ])

        ordinal_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ])

    elif mode == "tree":
        numerical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
        ])

        ordinal_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
        ])

    elif mode == "chi2":
        numerical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                MinMaxScaler(clip=True),
            ),
        ])

        ordinal_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "scaler",
                MinMaxScaler(clip=True),
            ),
        ])

    else:
        raise ValueError(
            "mode must be 'linear', 'tree', or 'chi2'"
        )

    return ColumnTransformer(
        transformers=[
            (
                "num",
                numerical_pipeline,
                numerical,
            ),
            (
                "ord",
                ordinal_pipeline,
                ordinal,
            ),
            (
                "cat",
                categorical_pipeline,
                categorical,
            ),
        ],
        remainder="drop",
        verbose_feature_names_out=True,
    )


### Pipeline purposes

- `linear`: Logistic Regression, SVM, and KNN.
- `tree`: Decision Tree, Random Forest, XGBoost, and tree-based stacking learners.
- `chi2`: Nonnegative transformed data for Chi-Square feature ranking.

All transformers will later be fitted only on the training portion of each validation fold.


## 8. Create a transformed-feature to original-feature map

In [12]:
def create_feature_map(
    fitted_preprocessor,
    dataset_name,
    mode,
):
    definition = FEATURE_DEFINITIONS[dataset_name]
    rows = []
    transformed_index = 0

    for transformer_name, role in [
        ("num", "numerical"),
        ("ord", "ordinal"),
    ]:
        columns = definition[role]

        for column in columns:
            rows.append({
                "dataset": dataset_name,
                "preprocessor_mode": mode,
                "transformed_index": transformed_index,
                "transformed_feature": (
                    f"{transformer_name}__{column}"
                ),
                "source_feature": column,
                "source_role": role,
            })
            transformed_index += 1

    categorical_columns = definition["categorical"]

    if categorical_columns:
        categorical_pipeline = (
            fitted_preprocessor.named_transformers_["cat"]
        )
        encoder = categorical_pipeline.named_steps["onehot"]
        encoded_names = encoder.get_feature_names_out(
            categorical_columns
        )

        name_index = 0

        for source_column, categories in zip(
            categorical_columns,
            encoder.categories_,
        ):
            for _ in categories:
                encoded_name = encoded_names[name_index]

                rows.append({
                    "dataset": dataset_name,
                    "preprocessor_mode": mode,
                    "transformed_index": transformed_index,
                    "transformed_feature": (
                        f"cat__{encoded_name}"
                    ),
                    "source_feature": source_column,
                    "source_role": "categorical",
                })

                transformed_index += 1
                name_index += 1

    return pd.DataFrame(rows)


## 9. Run one-fold leakage and transformation smoke tests

In [13]:
def preprocessing_smoke_test(
    dataset_name,
    dataframe,
):
    definition = FEATURE_DEFINITIONS[dataset_name]
    model_columns = get_model_columns(dataset_name)

    X = dataframe[model_columns].copy()
    y = dataframe["adverse_target"].copy()
    groups = dataframe["profile_group_id"].copy()

    splitter = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    train_index, test_index = next(
        splitter.split(X, y, groups)
    )

    X_train = X.iloc[train_index].copy()
    X_test = X.iloc[test_index].copy()
    y_train = y.iloc[train_index].copy()
    y_test = y.iloc[test_index].copy()

    train_groups = set(groups.iloc[train_index])
    test_groups = set(groups.iloc[test_index])
    shared_groups = train_groups.intersection(test_groups)

    assert len(shared_groups) == 0, (
        f"{dataset_name}: profile leakage was detected."
    )

    split_rows = [{
        "dataset": dataset_name,
        "fold": 1,
        "train_records": len(train_index),
        "test_records": len(test_index),
        "train_adverse_rate": y_train.mean(),
        "test_adverse_rate": y_test.mean(),
        "train_profile_groups": len(train_groups),
        "test_profile_groups": len(test_groups),
        "shared_profile_groups": len(shared_groups),
    }]

    test_rows = []
    feature_maps = []

    slug = (
        dataset_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    for mode in ["linear", "tree", "chi2"]:
        preprocessor = build_preprocessor(
            dataset_name,
            mode,
        )

        X_train_transformed = preprocessor.fit_transform(
            X_train,
            y_train,
        )

        X_test_transformed = preprocessor.transform(
            X_test
        )

        assert (
            X_train_transformed.shape[1]
            == X_test_transformed.shape[1]
        ), "Train/test transformed feature counts differ."

        assert np.isfinite(
            np.asarray(X_train_transformed)
        ).all(), "Non-finite training values detected."

        assert np.isfinite(
            np.asarray(X_test_transformed)
        ).all(), "Non-finite test values detected."

        if mode == "chi2":
            minimum_value = min(
                float(np.asarray(X_train_transformed).min()),
                float(np.asarray(X_test_transformed).min()),
            )
            assert minimum_value >= -1e-12, (
                "Chi-Square preprocessing produced negative values."
            )
        else:
            minimum_value = min(
                float(np.asarray(X_train_transformed).min()),
                float(np.asarray(X_test_transformed).min()),
            )

        feature_map = create_feature_map(
            preprocessor,
            dataset_name,
            mode,
        )

        assert (
            len(feature_map)
            == X_train_transformed.shape[1]
        ), "Feature-map size does not match transformed data."

        feature_map.to_csv(
            PREPROCESSING_DIR
            / f"{slug}_{mode}_feature_map.csv",
            index=False,
        )

        feature_maps.append(feature_map)

        joblib.dump(
            preprocessor,
            PIPELINE_DIR
            / f"{slug}_{mode}_smoke_test_preprocessor.joblib",
        )

        test_rows.append({
            "dataset": dataset_name,
            "preprocessor_mode": mode,
            "train_records": X_train_transformed.shape[0],
            "test_records": X_test_transformed.shape[0],
            "transformed_features": (
                X_train_transformed.shape[1]
            ),
            "minimum_transformed_value": minimum_value,
            "missing_train_values": int(
                np.isnan(
                    np.asarray(X_train_transformed)
                ).sum()
            ),
            "missing_test_values": int(
                np.isnan(
                    np.asarray(X_test_transformed)
                ).sum()
            ),
            "finite_values_confirmed": True,
            "profile_leakage_confirmed_absent": True,
        })

    return (
        pd.DataFrame(test_rows),
        pd.DataFrame(split_rows),
        pd.concat(feature_maps, ignore_index=True),
    )


all_smoke_tests = []
all_split_checks = []
all_feature_maps = []

for dataset_name, dataframe in DATASETS.items():
    smoke, split, feature_map = preprocessing_smoke_test(
        dataset_name,
        dataframe,
    )

    all_smoke_tests.append(smoke)
    all_split_checks.append(split)
    all_feature_maps.append(feature_map)

preprocessing_smoke_tests = pd.concat(
    all_smoke_tests,
    ignore_index=True,
)

split_leakage_checks = pd.concat(
    all_split_checks,
    ignore_index=True,
)

combined_feature_map = pd.concat(
    all_feature_maps,
    ignore_index=True,
)

preprocessing_smoke_tests.to_csv(
    PREPROCESSING_DIR / "preprocessing_smoke_tests.csv",
    index=False,
)

split_leakage_checks.to_csv(
    PREPROCESSING_DIR / "split_leakage_checks.csv",
    index=False,
)

combined_feature_map.to_csv(
    PREPROCESSING_DIR / "combined_transformed_feature_map.csv",
    index=False,
)

display(preprocessing_smoke_tests)
display(split_leakage_checks)


,dataset,preprocessor_mode,train_records,test_records,transformed_features,minimum_transformed_value,missing_train_values,missing_test_values,finite_values_confirmed,profile_leakage_confirmed_absent
0,Australian Credit Approval,linear,552,138,42,-1.519901,0,0,True,True
1,Australian Credit Approval,tree,552,138,42,0.000000,0,0,True,True
2,Australian Credit Approval,chi2,552,138,42,0.000000,0,0,True,True
3,German Credit,linear,800,200,61,-1.792775,0,0,True,True
4,German Credit,tree,800,200,61,0.000000,0,0,True,True
5,German Credit,chi2,800,200,61,0.000000,0,0,True,True
6,Taiwan Credit Card Default,linear,24001,5999,29,-6.338263,0,0,True,True
7,Taiwan Credit Card Default,tree,24001,5999,29,-339603.000000,0,0,True,True
8,Taiwan Credit Card Default,chi2,24001,5999,29,0.000000,0,0,True,True


,dataset,fold,train_records,test_records,train_adverse_rate,test_adverse_rate,train_profile_groups,test_profile_groups,shared_profile_groups
0,Australian Credit Approval,1,552,138,0.547101,0.586957,552,138,0
1,German Credit,1,800,200,0.291250,0.335000,800,200,0
2,Taiwan Credit Card Default,1,24001,5999,0.222282,0.216869,23956,5988,0


## 10. Save cleaned datasets and a validation protocol

In [14]:
for dataset_name, dataframe in DATASETS.items():
    slug = (
        dataset_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    dataframe.to_csv(
        PROCESSED_DATA_DIR / f"{slug}_cleaned.csv",
        index=False,
    )


validation_protocol = {
    "positive_class": (
        "1 = adverse, rejected, bad credit, or default"
    ),
    "negative_class": (
        "0 = favourable, approved, good credit, or non-default"
    ),
    "outer_validation": (
        "Repeated StratifiedGroupKFold: 5 folds x 5 repetitions"
    ),
    "outer_random_states": [
        42,
        142,
        242,
        342,
        442,
    ],
    "inner_validation": (
        "StratifiedGroupKFold with 5 folds inside each outer "
        "training partition"
    ),
    "grouping_rule": (
        "Identical complete predictor profiles share one "
        "profile_group_id and cannot cross fold boundaries."
    ),
    "leakage_control": [
        "Imputation fitted only on the training fold",
        "Encoding fitted only on the training fold",
        "Scaling fitted only on the training fold",
        "Feature selection fitted only on the training fold",
        "Class-imbalance treatment applied only to training data",
        "Hyperparameter selection performed only in inner validation",
        "Calibration and threshold selection performed only in inner validation",
    ],
}

with open(
    PREPROCESSING_DIR / "validation_protocol.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        validation_protocol,
        file,
        indent=4,
    )

print("Cleaned datasets saved to:", PROCESSED_DATA_DIR)
print("Validation protocol saved successfully.")


Cleaned datasets saved to: D:\PHD\Research Paper writing\3rd Obj. paper\data\processed
Validation protocol saved successfully.


## 11. Final Step 2 summary

In [15]:
cleaned_summary_rows = []

for dataset_name, dataframe in DATASETS.items():
    definition = FEATURE_DEFINITIONS[dataset_name]

    cleaned_summary_rows.append({
        "dataset": dataset_name,
        "records": len(dataframe),
        "model_predictors": len(
            get_model_columns(dataset_name)
        ),
        "categorical_predictors": len(
            definition["categorical"]
        ),
        "ordinal_predictors": len(
            definition["ordinal"]
        ),
        "numerical_predictors": len(
            definition["numerical"]
        ),
        "adverse_cases": int(
            dataframe["adverse_target"].sum()
        ),
        "adverse_rate": float(
            dataframe["adverse_target"].mean()
        ),
        "profile_groups": int(
            dataframe["profile_group_id"].nunique()
        ),
    })

cleaned_dataset_summary = pd.DataFrame(
    cleaned_summary_rows
)

cleaned_dataset_summary.to_csv(
    PREPROCESSING_DIR / "cleaned_dataset_summary.csv",
    index=False,
)

display(cleaned_dataset_summary)

print("\nFiles generated in preprocessing_protocol:")
for file_path in sorted(PREPROCESSING_DIR.iterdir()):
    print(" -", file_path.name)

print("\nStep 2 completed successfully.")


,dataset,records,model_predictors,categorical_predictors,ordinal_predictors,numerical_predictors,adverse_cases,adverse_rate,profile_groups
0,Australian Credit Approval,690,14,8,0,6,383,0.555072,690
1,German Credit,1000,20,13,0,7,300,0.300000,1000
2,Taiwan Credit Card Default,30000,23,3,6,14,6636,0.221200,29944



Files generated in preprocessing_protocol:
 - australian_credit_approval_chi2_feature_map.csv
 - australian_credit_approval_duplicate_profile_details.csv
 - australian_credit_approval_linear_feature_map.csv
 - australian_credit_approval_tree_feature_map.csv
 - cleaned_dataset_summary.csv
 - combined_transformed_feature_map.csv
 - duplicate_profile_summary.csv
 - german_credit_chi2_feature_map.csv
 - german_credit_duplicate_profile_details.csv
 - german_credit_linear_feature_map.csv
 - german_credit_tree_feature_map.csv
 - preprocessing_data_dictionary.csv
 - preprocessing_smoke_tests.csv
 - split_leakage_checks.csv
 - taiwan_category_consolidation.csv
 - taiwan_credit_card_default_chi2_feature_map.csv
 - taiwan_credit_card_default_duplicate_profile_details.csv
 - taiwan_credit_card_default_linear_feature_map.csv
 - taiwan_credit_card_default_tree_feature_map.csv
 - validation_protocol.json

Step 2 completed successfully.
